In [ ]:
%pip install -qU langchain_community pypdf
%pip install -q langchain

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/人物介紹英文版.pdf"
loader = PyPDFLoader(file_path)

In [ ]:
docs = loader.load()
document = docs[0].page_content

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=5,
    separators=["\n\n", "\n", ".", ";", ",", " "],
)
splitted_document = text_splitter.split_text(document)

In [ ]:
print(splitted_document)

['Zhang Zhikai, a 35-year-old data scientist and AI researcher from Taiwan, has a profound', 'passion for artificial intelligence and data analysis. During his undergraduate studies in', 'Computer Science at National Taiwan University, he actively participated in research', 'projects related to machine learning and natural language processing (NLP). His', 'graduation thesis explored the potential of deep learning in text classification applications.', 'In 2012, he pursued a Master’s degree in Machine Learning at Carnegie Mellon', 'University in the United States, focusing on reinforcement learning and knowledge graphs.', "His master's thesis integrated Graph Neural Networks (GNN) with reinforcement learning", 'techniques to enhance the efficiency of recommendation systems, and his research was', 'published at NeurIPS 2014.', 'After graduating, he joined Google Research as a Machine Learning Researcher, where he', 'developed NLP technologies based on Transformer models and contributed t

In [ ]:
%pip install chromadb

In [ ]:
%pip install -q langchain
%pip install -U langchain-groq
from google.colab import userdata

In [ ]:
import chromadb.utils.embedding_functions as embedding_functions
huggingface_ef = embedding_functions.HuggingFaceEmbeddingFunction(
    api_key=api_key,
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="collection",
    embedding_function = huggingface_ef,
    metadata={"hnsw:space": "cosine"}
)


In [ ]:
collection.add(
    documents = splitted_document,
    ids=[f'id{i}' for i in range(1, len(splitted_document)+1)]
)

In [ ]:
from functools import partial
from rich.console import Console
from rich.style import Style
from rich.theme import Theme

console = Console()
base_style = Style(color="#76B900", bold=True)
pprint = partial(console.print, style=base_style)

# 定義方法: 獲得在vector store裡面的文檔
def retrieve_vstore(query_text):
  results = collection.query(
      query_texts=query_text, # Chroma 會將這段chunk轉成向量
      n_results=5, # 回傳五個結果
      include=["documents", "distances"],
  )
  return results['documents'][0]

# 測試: 獲得在vector store裡面的文檔的回傳樣貌
results = collection.query(
    query_texts=['who is he'], 
    n_results=5,
    include=["documents", "distances"],
)
pprint(results)
print("----------------------------------------")
pprint(results['documents'][0])

{
    'ids': [['id2', 'id28', 'id3', 'id1', 'id14']],
    'embeddings': None,
    'documents': [
        [
            'passion for artificial intelligence and data analysis. During his undergraduate studies in',
            'learning, and model interpretability. He is proficient in deep learning frameworks such as',
            'Computer Science at National Taiwan University, he actively participated in research',
            'Zhang Zhikai, a 35-year-old data scientist and AI researcher from Taiwan, has a profound',
            'Google, he published several influential papers on Large Language Models (LLM) and'
        ]
    ],
    'uris': None,
    'data': None,
    'metadatas': None,
    'distances': [
        [0.5360561609268188, 0.553363561630249, 0.6660404205322266, 0.675655722618103, 0.6902749538421631]
    ],
    'included': [<IncludeEnum.distances: 'distances'>, <IncludeEnum.documents: 'documents'>]
}

----------------------------------------


[
    'passion for artificial intelligence and data analysis. During his undergraduate studies in',
    'learning, and model interpretability. He is proficient in deep learning frameworks such as',
    'Computer Science at National Taiwan University, he actively participated in research',
    'Zhang Zhikai, a 35-year-old data scientist and AI researcher from Taiwan, has a profound',
    'Google, he published several influential papers on Large Language Models (LLM) and'
]

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="mixtral-8x7b-32768",
    temperature=0.0,
    max_retries=2,
    api_key=userdata.get('groq')
)

In [ ]:
# 測試llm串接是否成功
input_text = "The meaning of life is "
llm.invoke(input_text)

AIMessage(content='The meaning of life is a philosophical and metaphysical question related to the purpose or significance of life or existence in general. This concept has been approached by many perspectives including religious, philosophical, and scientific viewpoints.\n\nSome people believe that the meaning of life is to seek happiness, knowledge, or personal fulfillment, while others believe that it is to serve a higher power or to contribute to the betterment of humanity.\n\nUltimately, the meaning of life is a deeply personal and subjective concept that varies from person to person. It is up to each individual to determine their own purpose and meaning in life based on their values, beliefs, and experiences.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 141, 'prompt_tokens': 13, 'total_tokens': 154, 'completion_time': 0.217040996, 'prompt_time': 0.001998662, 'queue_time': 0.019109835, 'total_time': 0.219039658}, 'model_name': 'mixtral-8x7b-32768

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.runnables.passthrough import RunnableAssign
from operator import itemgetter


chat_prompt = ChatPromptTemplate.from_messages([("system",
    "You are a document chatbot. Help the user as they ask questions about documents."
    " User messaged just asked: {input}\n\n"
    " From this, we have retrieved the following potentially-useful info: "
    " Document Retrieval:\n{context}\n\n"
    " (Answer only from retrieval. Only cite sources that are used. Make your response conversational.)"
), ('user', '{input}')])

chain = (
    RunnableAssign({'context': lambda q: RunnableLambda(itemgetter('input')) | retrieve_vstore})
    | chat_prompt
    | llm
    | StrOutputParser()

)
query = input('input your question: ')
pprint(chain.invoke({'input': query}))

input your questionwho is he


The person being referred to is Zhang Zhikai, a 35-year-old data scientist and AI researcher from Taiwan. He has a 
passion for artificial intelligence and data analysis, and he studied Computer Science at National Taiwan 
University. During his undergraduate studies, he actively participated in research and published several 
influential papers on Large Language Models (LLM) while at Google. He is also proficient in deep learning 
frameworks.